In [8]:
import polars as pl
from wjp_judicial_independence.config import PATH_DATA_RAW, PATH_DATA_INTERIM
from wjp_judicial_independence.sentiment import classify_sentiment

MODULE1_STRATEGY = "embeddings" # "embeddings" | "llm" | "llm-api"

df = pl.read_parquet(PATH_DATA_INTERIM/f"module1/df_{MODULE1_STRATEGY}_strategy_judicial_independence.parquet").filter(pl.col("is_judicial_independence"))
df

country,pillar,impact,event,embedding,score_compliance_with_judicial_decisions,score_judicial_corruption_scandals,score_judicial_appointments_and_public_trust,is_judicial_independence
str,str,str,str,"array[f32, 768]",f32,f32,f32,bool
"""POLAND""","""Constraints on Government Powe…","""Very Positive""","""* **Challenging Unlawful Actio…","[0.055642, 0.030804, … 0.019532]",0.502467,0.640566,0.535574,true
"""POLAND""","""Constraints on Government Powe…","""Very Positive""","""* **Resisting Political Interf…","[0.043142, 0.027225, … 0.021994]",0.575841,0.648641,0.655584,true
"""POLAND""","""Constraints on Government Powe…","""Very Positive""","""* **Combating Corruption and F…","[-0.020327, 0.069766, … 0.025625]",0.375987,0.548091,0.33353,true
"""POLAND""","""Constraints on Government Powe…","""Positive""","""* **Financial Misconduct:** Th…","[0.029335, 0.031608, … 0.015644]",0.341157,0.617028,0.345667,true
"""POLAND""","""Constraints on Government Powe…","""Positive""","""* **Sentencing Former CBA Head…","[0.05477, 0.029898, … 0.008184]",0.438802,0.519161,0.423901,true
…,…,…,…,…,…,…,…,…
"""ITALY""","""Criminal Justice""","""Very Negative""","""* **Lack of Resources and Expe…","[0.038798, 0.07713, … 0.027266]",0.374488,0.528867,0.389815,true
"""ITALY""","""Criminal Justice""","""Very Negative""","""* **Mishandling of Evidence:**…","[0.049785, 0.062276, … 0.00493]",0.469431,0.53739,0.368716,true
"""ITALY""","""Criminal Justice""","""Very Negative""","""* **Failure to Protect Witness…","[0.025998, 0.036977, … 0.000316]",0.405182,0.507332,0.421322,true


# Recuperación de Tópicos Relevantes

## BERTopic: Embeddings + Reducción de dimensionalidad + Clusterización + CountVectorizer + TFIDF + FineTuning

Esta arquitectura se escogió de la literatura: Grootendorst, M. (2022). *BERTopic: Neural topic modeling with a class-based TF-IDF procedure*. arXiv preprint arXiv:2203.05794.

![image.png](../assets/bertopic.svg)

In [9]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI
from bertopic.vectorizers import ClassTfidfTransformer

# Bloque 1 - Embeddings
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Bloque 2 - Reducción de dimensionalidad
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

# Bloque 3 - Clusterización
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

# Bloque 4 - CountVectorizer
custom_stops = list(ENGLISH_STOP_WORDS) + [
    "article", "articles", "discussing", "discussed",
    "regarding", "concerning", "country", "report",
    "says", "said", "according", "also", "would",
    "could", "may", "might", "shall", "upon",
    "one", "two", "new", "made", "like",
    "including", "related", "based", "using",
    "italy", "italian", "polish", "poland", "hungary", "hungarian"
]

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=0.01,
    max_df=0.80,
    stop_words=custom_stops,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
)

# Bloque 5 - Representación de topico con TF-IDF
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Modelo general para comparar paises

In [10]:
# general
# Bloque 6 - Finetuning de representación de tópicos
representation_model = [KeyBERTInspired(top_n_words=15), MaximalMarginalRelevance(diversity=0.7)]

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    nr_topics=8
)
topics, probs = topic_model.fit_transform(df["event"])
topic_model.save(PATH_DATA_INTERIM/f"module2/topic_modelling/bertopic_general_m1_{MODULE1_STRATEGY}", serialization="safetensors", save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_NAME)

### Modelo detallado por pais

In [11]:
representation_model = [KeyBERTInspired(top_n_words=40), MaximalMarginalRelevance(diversity=0.6)]

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
)

for (country, ), df_grouped in df.group_by("country"):
    r = topic_model.fit_transform(df_grouped["event"])
    topic_model.save(PATH_DATA_INTERIM/f"module2/topic_modelling/bertopic_{country}_m1_{MODULE1_STRATEGY}", serialization="safetensors", save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_NAME)